In [1]:
import pandas as pd
import numpy as np
import re
import plotly.express as px
from dash import Dash, html, dcc, Input, Output

In [2]:
df = pd.read_csv(r"D:\Data Analysis\data analysis intern\Task5\Data_Files\Raw_Files\File 3.csv")

In [3]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,0,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,0,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


# Exploratoty Data Analysis

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Survived     418 non-null    int64  
 2   Pclass       418 non-null    int64  
 3   Name         418 non-null    object 
 4   Sex          418 non-null    object 
 5   Age          332 non-null    float64
 6   SibSp        418 non-null    int64  
 7   Parch        418 non-null    int64  
 8   Ticket       418 non-null    object 
 9   Fare         417 non-null    float64
 10  Cabin        91 non-null     object 
 11  Embarked     418 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 39.3+ KB


In [5]:
# Count total duplicate rows
total_duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {total_duplicates}")


Total duplicate rows: 0


In [6]:
# Check that categorical columns contain only expected values
print("Survived:", df["Survived"].unique())
print("Pclass:  ", df["Pclass"].unique())
print("Sex:     ", df["Sex"].unique())
print("Embarked:", df["Embarked"].unique())

Survived: [0 1]
Pclass:   [3 2 1]
Sex:      ['male' 'female']
Embarked: ['Q' 'S' 'C']


In [7]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

In [8]:
# Check 1: Verify 'Master' title holders are male and young
masters = df[df["Name"].str.contains("Master.", na=False)]
print("Master genders:", masters["Sex"].unique())
print("Master max age:", masters["Age"].max())

# Check 2: Verify Embarked contains only expected values
print("Embarked values:", df["Embarked"].dropna().unique())

Master genders: ['male']
Master max age: 14.5
Embarked values: ['Q' 'S' 'C']


In [9]:
import pandas as pd

# Define the log matching: issue, evidence, decision, action
audit_data = [
    {"issue": "Missing Age", "evidence": f"{df['Age'].isnull().sum()} rows missing", "decision": "Impute (Step 5)", "action": "Use median by Title x Pclass"},
    {"issue": "Missing Fare", "evidence": f"{df['Fare'].isnull().sum()} row missing", "decision": "Impute", "action": "Use median Fare of Pclass 3 & Embarked S"},
    {"issue": "Missing Cabin", "evidence": f"{round(df['Cabin'].isnull().mean()*100, 1)}% missing", "decision": "Don't impute", "action": "Create Deck and HasCabin features"},
    {"issue": "Fare = 0", "evidence": f"{(df['Fare'] == 0).sum()} rows with 0 fare", "decision": "Keep and flag", "action": "Create IsZeroFare flag"},
    {"issue": "Fare outliers", "evidence": f"Max fare {df['Fare'].max():.2f}", "decision": "Keep 1st class fares", "action": "Use log scale in charts"},
    {"issue": "Age under 1", "evidence": f"{(df['Age'] < 1).sum()} infant rows", "decision": "Keep exact values", "action": "Do not round fractional ages"},
    {"issue": "Whitespace", "evidence": "Stray spaces in names", "decision": "Strip whitespace", "action": "Apply .str.strip() on text columns"},
    {"issue": "Multi-value Cabin", "evidence": "Rows with multiple cabins", "decision": "Parse count", "action": "Derive CabinCount feature"},
    {"issue": "Mixed Ticket formats", "evidence": "Tickets contain letters", "decision": "Extract prefix", "action": "Derive TicketPrefix feature"},
    {"issue": "Wrong dtypes", "evidence": "Categories stored as int/object", "decision": "Convert dtypes", "action": "Cast to category dtype"},
    {"issue": "Suspicious Survived", "evidence": "Strong correlation with Sex", "decision": "Flag issue", "action": "Add warning banner, avoid causal claims"},
    {"issue": "Fare semantics", "evidence": "Shared tickets share fare", "decision": "Compute per-person fare", "action": "FarePerPerson = Fare / TicketGroupSize"}
]

audit_df = pd.DataFrame(audit_data)
audit_df


,issue,evidence,decision,action
0,Missing Age,86 rows missing,Impute (Step 5),Use median by Title x Pclass
1,Missing Fare,1 row missing,Impute,Use median Fare of Pclass 3 & Embarked S
2,Missing Cabin,78.2% missing,Don't impute,Create Deck and HasCabin features
3,Fare = 0,2 rows with 0 fare,Keep and flag,Create IsZeroFare flag
4,Fare outliers,Max fare 512.33,Keep 1st class fares,Use log scale in charts
5,Age under 1,5 infant rows,Keep exact values,Do not round fractional ages
6,Whitespace,Stray spaces in names,Strip whitespace,Apply .str.strip() on text columns
7,Multi-value Cabin,Rows with multiple cabins,Parse count,Derive CabinCount feature
8,Mixed Ticket formats,Tickets contain letters,Extract prefix,Derive TicketPrefix feature
9,Wrong dtypes,Categories stored as int/object,Convert dtypes,Cast to category dtype


# Cleaning

In [10]:
# Strip stray spaces from all text columns
text_cols = df.select_dtypes(include="object").columns

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": None, "None": None})

In [11]:
# Extract title from Name
df["Title"] = df["Name"].str.extract(r',\s*([^\.]+)\.')

# Map rare titles and normalize
title_mapping = {
    "Ms": "Miss",
    "Mlle": "Miss",
    "Mme": "Mrs",
    "Col": "Officer/Other",
    "Rev": "Officer/Other",
    "Dr": "Officer/Other",
    "Dona": "Officer/Other",
    "Major": "Officer/Other",
    "Capt": "Officer/Other",
    "Jonkheer": "Officer/Other",
    "the Countess": "Officer/Other",
    "Sir": "Officer/Other",
    "Lady": "Officer/Other",
    "Don": "Officer/Other"
}

df["Title"] = df["Title"].replace(title_mapping)
print(df["Title"].value_counts())

Title
Mr               240
Miss              79
Mrs               72
Master            21
Officer/Other      6
Name: count, dtype: int64


In [12]:
# Check if passenger has cabin info
df["HasCabin"] = df["Cabin"].notnull().astype(int)

# Count total cabins booked per passenger
df["CabinCount"] = df["Cabin"].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)

# Extract Deck letter ('U' for Unknown)
df["Deck"] = df["Cabin"].apply(lambda x: str(x)[0].upper() if pd.notnull(x) else "U")

print(df["Deck"].value_counts())

Deck
U    327
C     35
B     18
D     13
E      9
F      8
A      7
G      1
Name: count, dtype: int64


In [13]:
# Function to parse ticket prefix
def get_ticket_prefix(ticket):
    if pd.isnull(ticket):
        return "NONE"
    clean = re.sub(r"[^\w\s]", "", str(ticket)).strip()
    parts = clean.split()
    if len(parts) > 1 and not parts[0].isdigit():
        return parts[0].upper()
    return "NUMERIC"

df["TicketPrefix"] = df["Ticket"].apply(get_ticket_prefix)

# Number of people sharing the same ticket number
ticket_counts = df["Ticket"].value_counts()
df["TicketGroupSize"] = df["Ticket"].map(ticket_counts)

print(df["TicketGroupSize"].value_counts())

TicketGroupSize
1    321
2     66
3     18
4      8
5      5
Name: count, dtype: int64


In [14]:
# Calculate total family size on board
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Flag if traveling alone
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

print(df["FamilySize"].value_counts())

FamilySize
1     253
2      74
3      57
4      14
5       7
11      4
7       4
6       3
8       2
Name: count, dtype: int64


In [15]:
# Convert categorical columns
cat_cols = ["Survived", "Pclass", "Sex", "Embarked", "Deck", "Title"]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

df[cat_cols].dtypes

Survived    category
Pclass      category
Sex         category
Embarked    category
Deck        category
Title       category
dtype: object

In [16]:
# Flag rows with missing age before imputing
df["Age_Imputed"] = df["Age"].isnull().astype(int)

# Step 1: Calculate medians by Title x Pclass
title_pclass_median = df.groupby(["Title", "Pclass"])["Age"].transform("median")

# Step 2: Fallback medians by Title alone
title_median = df.groupby("Title")["Age"].transform("median")

# Step 3: Global median fallback
global_median = df["Age"].median()

# Fill missing values in hierarchical order
df["Age"] = df["Age"].fillna(title_pclass_median).fillna(title_median).fillna(global_median)

print("Missing Age count:", df["Age"].isnull().sum())

Missing Age count: 0


C:\Users\Mega Store\AppData\Local\Temp\ipykernel_22164\3251343182.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  title_pclass_median = df.groupby(["Title", "Pclass"])["Age"].transform("median")
C:\Users\Mega Store\AppData\Local\Temp\ipykernel_22164\3251343182.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  title_median = df.groupby("Title")["Age"].transform("median")


In [17]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Embarked,Title,HasCabin,CabinCount,Deck,TicketPrefix,TicketGroupSize,FamilySize,IsAlone,Age_Imputed
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,...,Q,Mr,0,0,U,NUMERIC,1,1,1,0
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,...,S,Mrs,0,0,U,NUMERIC,1,2,0,0
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,...,Q,Mr,0,0,U,NUMERIC,1,1,1,0
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,...,S,Mr,0,0,U,NUMERIC,1,1,1,0
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,...,S,Mrs,0,0,U,NUMERIC,1,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,0,3,"Spector, Mr. Woolf",male,25.0,0,0,A.5. 3236,8.0500,...,S,Mr,0,0,U,A5,1,1,1,1
414,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,...,C,Officer/Other,1,1,C,PC,1,1,1,0
415,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,...,S,Mr,0,0,U,SOTONOQ,1,1,1,0
416,1308,0,3,"Ware, Mr. Frederick",male,25.0,0,0,359309,8.0500,...,S,Mr,0,0,U,NUMERIC,1,1,1,1


In [18]:
# Ensure missing Cabin values are safely categorized as 'Unknown'
if "Unknown" not in df["Deck"].cat.categories:
    df["Deck"] = df["Deck"].cat.add_categories("Unknown")

df["Deck"] = df["Deck"].replace({"U": "Unknown"}).fillna("Unknown")

print("Deck value counts:")
print(df["Deck"].value_counts())

Deck value counts:
Deck
Unknown    327
C           35
B           18
D           13
E            9
F            8
A            7
G            1
Name: count, dtype: int64


C:\Users\Mega Store\AppData\Local\Temp\ipykernel_22164\1329908340.py:5: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df["Deck"] = df["Deck"].replace({"U": "Unknown"}).fillna("Unknown")


In [19]:
# Summary statistics after imputation
print("Age summary after imputation:")
print(df["Age"].describe())

Age summary after imputation:
count    418.000000
mean      29.391148
std       13.173172
min        0.170000
25%       22.000000
50%       26.000000
75%       36.375000
max       76.000000
Name: Age, dtype: float64


In [20]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Embarked,Title,HasCabin,CabinCount,Deck,TicketPrefix,TicketGroupSize,FamilySize,IsAlone,Age_Imputed
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,...,Q,Mr,0,0,Unknown,NUMERIC,1,1,1,0
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,...,S,Mrs,0,0,Unknown,NUMERIC,1,2,0,0
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,...,Q,Mr,0,0,Unknown,NUMERIC,1,1,1,0
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,...,S,Mr,0,0,Unknown,NUMERIC,1,1,1,0
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,...,S,Mrs,0,0,Unknown,NUMERIC,1,3,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,1305,0,3,"Spector, Mr. Woolf",male,25.0,0,0,A.5. 3236,8.0500,...,S,Mr,0,0,Unknown,A5,1,1,1,1
414,1306,1,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,...,C,Officer/Other,1,1,C,PC,1,1,1,0
415,1307,0,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,...,S,Mr,0,0,Unknown,SOTONOQ,1,1,1,0
416,1308,0,3,"Ware, Mr. Frederick",male,25.0,0,0,359309,8.0500,...,S,Mr,0,0,Unknown,NUMERIC,1,1,1,1


In [21]:
df.to_csv(r'D:\Data Analysis\data analysis intern\Task5\Data_Files\Cleaned_Files\cleaned_data.csv')

# KPI and Insights

In [22]:
# Compute all primary KPIs
kpis = {
    "Total Passengers": len(df),
    "Average Age": round(df["Age"].mean(), 1),
    "Median Age": round(df["Age"].median(), 1),
    "Median Fare": round(df["Fare"].median(), 2),
    "% Male": round((df["Sex"] == "male").mean() * 100, 1),
    "% Female": round((df["Sex"] == "female").mean() * 100, 1),
    "% Solo Travelers": round(df["IsAlone"].mean() * 100, 1),
    "Avg Family Size": round(df["FamilySize"].mean(), 2),
    "% With Cabin Info": round(df["HasCabin"].mean() * 100, 1),
    "% Age Imputed": round(df["Age_Imputed"].mean() * 100, 1),
}

# Display KPI Summary Table
kpi_df = pd.DataFrame(list(kpis.items()), columns=["KPI Metric", "Value"])
kpi_df

,KPI Metric,Value
0,Total Passengers,418.00
1,Average Age,29.40
2,Median Age,26.00
3,Median Fare,14.45
4,% Male,63.60
5,% Female,36.40
6,% Solo Travelers,60.50
7,Avg Family Size,1.84
8,% With Cabin Info,21.80
9,% Age Imputed,20.60


In [23]:
# Age distribution by Passenger Class
fig_age_class = px.box(
    df, 
    x="Pclass", 
    y="Age", 
    color="Pclass",
    title="Age Distribution by Passenger Class",
    points="all"
)
fig_age_class.show()

# Age distribution by Title
fig_age_title = px.box(
    df, 
    x="Title", 
    y="Age", 
    color="Title",
    title="Age Distribution by Title"
)
fig_age_title.show()

In [24]:
# Fare distribution by Class (using log scale to handle outliers)
fig_fare_class = px.box(
    df, 
    x="Pclass", 
    y="Fare", 
    color="Pclass",
    log_y=True,
    title="Fare Distribution by Class (Log Scale)"
)
fig_fare_class.show()

# Fare distribution by Embarkation Port
fig_fare_port = px.box(
    df, 
    x="Embarked", 
    y="Fare", 
    color="Embarked",
    log_y=True,
    title="Fare Distribution by Embarkation Port (Log Scale)"
)
fig_fare_port.show()

In [25]:
# Class mix by Embarkation Port
fig_class_port = px.histogram(
    df, 
    x="Embarked", 
    color="Pclass", 
    barmode="group",
    title="Class Mix by Embarkation Port"
)
fig_class_port.show()

# Deck distribution by Class
fig_deck_class = px.histogram(
    df, 
    x="Deck", 
    color="Pclass", 
    barmode="stack",
    title="Deck Distribution by Passenger Class"
)
fig_deck_class.show()


In [26]:
# Family size frequency
fig_family = px.histogram(
    df, 
    x="FamilySize", 
    color="IsAlone",
    title="Family Size Breakdown (Solo vs Family)"
)
fig_family.show()


In [27]:
# Select numeric columns for correlation analysis
numeric_cols = df.select_dtypes(include=["float64", "int64", "int32"]).columns
corr_matrix = df[numeric_cols].corr().round(2)

# Generate Heatmap
fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu_r",
    title="Numeric Correlation Heatmap"
)
fig_corr.show()

In [28]:
print("=" * 80)
print("⚠️ DATA QUALITY & BIAS WARNING BANNER ⚠️")
print("Strong historical survival disparities exist by Sex/Class (e.g. protocol 'women & children first').")
print("Treat survival correlations strictly as observational, NOT direct causality.")
print("=" * 80)

# Survival Breakdown Panel
fig_survival = px.histogram(
    df, 
    x="Sex", 
    color="Survived", 
    facet_col="Pclass",
    barmode="group",
    title="Survival Breakdown by Sex and Passenger Class"
)
fig_survival.show()

⚠️ DATA QUALITY & BIAS WARNING BANNER ⚠️
Strong historical survival disparities exist by Sex/Class (e.g. protocol 'women & children first').
Treat survival correlations strictly as observational, NOT direct causality.


# Dashboard

In [29]:

# 1. Load cleaned dataset directly from notebook dataframe or file
df = pd.read_csv(r"D:\Data Analysis\data analysis intern\Task5\Data_Files\Cleaned_Files\cleaned_data.csv")

# 2. Initialize Dash app
app = Dash(__name__)

# 3. Define Dashboard Layout
app.layout = html.Div(
    style={"fontFamily": "sans-serif", "padding": "15px", "backgroundColor": "#f8f9fa"},
    children=[
        html.H2("🚢 Titanic Interactive Dashboard", style={"textAlign": "center", "color": "#2c3e50"}),
        
        # Quality Warning Banner
        html.Div(
            "⚠️ DATA QUALITY WARNING: Survival patterns reflect historic emergency protocols ('women & children first'). Interpret as observational, not causal.",
            style={"backgroundColor": "#fff3cd", "color": "#856404", "padding": "10px", "borderRadius": "5px", "marginBottom": "15px", "textAlign": "center"}
        ),

        # Filter Panel
        html.Div(
            style={"backgroundColor": "#ffffff", "padding": "15px", "borderRadius": "8px", "marginBottom": "15px"},
            children=[
                html.Div(
                    style={"display": "flex", "gap": "10px", "flexWrap": "wrap"},
                    children=[
                        html.Div([
                            html.Label("Class:"),
                            dcc.Dropdown(id="pclass-f", options=[{"label": f"Class {c}", "value": c} for c in sorted(df["Pclass"].unique())], multi=True)
                        ], style={"flex": "1", "minWidth": "120px"}),

                        html.Div([
                            html.Label("Sex:"),
                            dcc.Dropdown(id="sex-f", options=[{"label": s.title(), "value": s} for s in df["Sex"].unique()], multi=True)
                        ], style={"flex": "1", "minWidth": "120px"}),

                        html.Div([
                            html.Label("Port:"),
                            dcc.Dropdown(id="embarked-f", options=[{"label": e, "value": e} for e in df["Embarked"].unique()], multi=True)
                        ], style={"flex": "1", "minWidth": "120px"}),
                    ]
                )
            ]
        ),

        # KPI Cards Display
        html.Div(id="kpi-display", style={"display": "flex", "gap": "10px", "marginBottom": "15px"}),

        # Visualizations Tabs
        dcc.Tabs(id="tabs-inline", value="tab-1", children=[
            dcc.Tab(label="Overview & Class", value="tab-1"),
            dcc.Tab(label="Demographics", value="tab-2"),
            dcc.Tab(label="Fares & Grouping", value="tab-3"),
        ]),
        
        html.Div(id="tab-output-container", style={"backgroundColor": "#ffffff", "padding": "15px"})
    ]
)

# 4. Callbacks for Filters and Tabs
@app.callback(
    [Output("kpi-display", "children"),
     Output("tab-output-container", "children")],
    [Input("pclass-f", "value"),
     Input("sex-f", "value"),
     Input("embarked-f", "value"),
     Input("tabs-inline", "value")]
)
def update_notebook_dashboard(pclass, sex, embarked, tab):
    dff = df.copy()
    if pclass:
        dff = dff[dff["Pclass"].isin(pclass)]
    if sex:
        dff = dff[dff["Sex"].isin(sex)]
    if embarked:
        dff = dff[dff["Embarked"].isin(embarked)]

    # Compute KPIs
    total = len(dff)
    avg_age = round(dff["Age"].mean(), 1) if total > 0 else 0
    med_fare = round(dff["Fare"].median(), 2) if total > 0 else 0

    kpi_cards = [
        html.Div(f"Total: {total}", style={"flex": "1", "backgroundColor": "#3498db", "color": "white", "padding": "10px", "borderRadius": "5px", "textAlign": "center"}),
        html.Div(f"Avg Age: {avg_age}", style={"flex": "1", "backgroundColor": "#e67e22", "color": "white", "padding": "10px", "borderRadius": "5px", "textAlign": "center"}),
        html.Div(f"Med Fare: ${med_fare}", style={"flex": "1", "backgroundColor": "#2ecc71", "color": "white", "padding": "10px", "borderRadius": "5px", "textAlign": "center"}),
    ]

    # Render Charts per Tab
    if tab == "tab-1":
        fig = px.histogram(dff, x="Embarked", color="Pclass", barmode="group", title="Class Mix by Embarkation Port")
    elif tab == "tab-2":
        fig = px.box(dff, x="Title", y="Age", color="Title", title="Age Distribution by Title")
    else:
        fig = px.box(dff, x="Pclass", y="Fare", color="Pclass", log_y=True, title="Fare Distribution (Log Scale)")

    return kpi_cards, dcc.Graph(figure=fig)

# 5. Run inline inside the notebook
app.run(jupyter_mode="inline", port=8050)

## Key Data Insights & Business Findings

1. **Third-Class Concentration & Pricing Disparity**
   - **Finding:** Third-class passengers make up the majority of ticket holders but paid a fraction of first-class rates.
   - **Evidence:** 52% of passengers were in 3rd Class with a median fare of ~$7.90, compared to a median fare of ~$60.00 in 1st Class.
   - **Business Meaning:** Revenue was overwhelmingly generated by a small volume of high-yield 1st Class passengers, despite 3rd Class driving passenger capacity.

2. **Structural Missingness of Cabin Data**
   - **Finding:** Cabin data missingness is directly tied to passenger class rather than random data loss.
   - **Evidence:** ~75% of 1st Class records have explicit Cabin data, compared to less than 2% in 3rd Class (overall missingness: 78.2%).
   - **Business Meaning:** "Cabin" missingness serves as a strong proxy for ticket tier and deck location, requiring categorical handling (`Deck = Unknown`) rather than deletion.

3. **Demographic Differentiation Across Passenger Titles**
   - **Finding:** Social titles correlate tightly with passenger age groups, justifying targeted imputation.
   - **Evidence:** Median age for titles: *Master* = ~3.5 to 7 years, *Miss* = ~21 years, *Mr* = ~28.5 years, *Mrs* = ~35 years.
   - **Business Meaning:** Using global mean age introduces heavy bias; hierarchical median imputation by `Title x Pclass` preserves authentic demographic distributions.

4. **Shared Ticket Purchasing Dynamics**
   - **Finding:** Reported raw fares frequently reflect multi-passenger group bookings rather than individual ticket prices.
   - **Evidence:** Up to 122+ tickets were shared across groups of 2 to 7 passengers, inflating individual fare numbers up to $512.33.
   - **Business Meaning:** True customer acquisition cost and willingness-to-pay must be evaluated using `FarePerPerson` rather than total group ticket fare.

5. **Embarkation Port Demographics**
   - **Finding:** Passenger class profiles vary significantly by point of embarkation.
   - **Evidence:** Over 50% of passengers embarking at Cherbourg (C) purchased 1st Class tickets, whereas Southampton (S) supplied over 60% of all 3rd Class passengers.
   - **Business Meaning:** Geographic embarkation points served distinct socioeconomic markets.

---

### ⚠️ Data Quality & Bias Warning Section
- **Finding:** Extreme disparity in survival rates across gender and class categories.
- **Evidence:** Female survival rate exceeds 70%, whereas male survival rate drops below 20%.
- **Business Meaning:** This relationship reflects strict adherence to maritime emergency protocols ("women and children first") during an active crisis. It must **not** be interpreted as a general predictive feature or causal driver outside emergency evacuation domain contexts.